In [1]:
import sqlite3
import pandas as pd
df1 = pd.read_excel('C:/Users/white/OneDrive/Desktop/docs/Sample - Superstore.xls', sheet_name=0)
df2 = pd.read_excel('C:/Users/white/OneDrive/Desktop/docs/Sample - Superstore.xls', sheet_name=1)
df3 = pd.read_excel('C:/Users/white/OneDrive/Desktop/docs/Sample - Superstore.xls', sheet_name=2)
df4 = pd.read_csv('C:/Users/white/OneDrive/Desktop/docs/Social_Network_Ads.csv')

# Load data table in SQLite
conn = sqlite3.connect('Chinook.db')
df1.to_sql('store', conn, if_exists='replace', index=False) 
df2.to_sql('store_return', conn, if_exists='replace', index=False) 
df3.to_sql('store_employee', conn, if_exists='replace', index=False) 
df4.to_sql('store_marketing', conn, if_exists='replace', index=False) 

C:\Users\white\AppData\Local\Temp\ipykernel_29872\3080706515.py:6: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df4 = pd.read_csv('C:/Users/white/OneDrive/Desktop/docs/Social_Network_Ads.csv')


1048575

In [2]:
import os
import sqlite3
import pandas as pd
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate 
from langchain_community.utilities import SQLDatabase
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.agent_toolkits import create_sql_agent
from langchain_core.output_parsers import StrOutputParser
from langchain.agents.agent_types import AgentType
from pydantic import BaseModel, Field
from dotenv import load_dotenv
load_dotenv()  # LangChain will automatically pick up the GOOGLE_API_KEY from the environment

api_key_1 = os.getenv("GEMINI_API_KEY_1")
api_key_2 = os.getenv("GEMINI_API_KEY_2")

model = ChatGoogleGenerativeAI(model='gemini-2.5-flash-lite', google_api_key = api_key_1,temperature=0)
model_flash1 = ChatGoogleGenerativeAI(model='gemini-2.5-flash', google_api_key = api_key_1,temperature=0)
model_flash2 = ChatGoogleGenerativeAI(model='gemini-2.5-flash', google_api_key = api_key_2,temperature=0)

# Connect to the SQLite database
db = SQLDatabase.from_uri("sqlite:///Chinook.db")

## Process
- Fetch DB schema
- Prepare a prompt to give System instruction for SQL query and use DB schema to generate SQL query
- Include column level values like value/date range, variables (5% or 8% repetation)
- Apply standard query translation methods: Multi-Query, RAG-FUsion, Decomposition, Step-back
- In case of multiple question (breakdown problem into samll query) then prepare a combination of Question & SQL Query
- Identify is it SQL code or python code
- Prepare a function to execute python code for numbers, table and graph
- Prepare a function to execute the SQL code
- Prepare a function to pass output
- prepare function to validate the output
- prepare chain = Prompt | agent_executor  (later will add more nodes)

In [3]:
# Fetching existing tables list in SQlite DB
db_file = "Chinook.db"
sql_query = "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%';" # Execute the query to get all user-defined tables

# Using python
db_tables = []
try:
    with sqlite3.connect(db_file) as conn:
        cursor = conn.cursor()
        cursor.execute(sql_query)
        tables = cursor.fetchall()
        # print(tables)

        if not tables:
            print("No user-defined tables found.")
        else:
            print("Existing tables:")
            for table in tables:
                db_tables.append(table[0])
                print(table[0]) # The table name is the first element of each tuple
except sqlite3.Error as e:
    print(f"An error occurred: {e}")

Existing tables:
store
store_return
store_employee
store_marketing


In [4]:
# Table detais function:

import pandas as pd
import numpy as np

def get_column_details(df: pd.DataFrame) -> dict:

    column_details = {}
    total_rows = len(df)
    
    for col in df.columns:
        #column_details[col]['Column Name'] = col
        column_details[col] = {} 
        
        # Handle numerical columns
        if pd.api.types.is_numeric_dtype(df[col]):
            col_min = df[col].min()
            col_max = df[col].max()
            column_details[col]['type'] = 'numerical'
            column_details[col]['range'] = (col_min, col_max)
            
        # Handle date columns
        elif pd.api.types.is_datetime64_any_dtype(df[col]):
            col_start = df[col].min()
            col_end = df[col].max()
            column_details[col]['type'] = 'date'
            column_details[col]['date_range'] = (col_start, col_end)
            
        # Handle categorical columns
        else:
            unique_count = df[col].nunique()
            # Calculate the 5% threshold
            if total_rows < 10:
                threshold = 0.30 * total_rows
            elif 10 <= total_rows < 100:
                threshold = 0.30 * total_rows
            elif 100 <= total_rows < 1000:
                threshold = 0.10 * total_rows
            elif 1000 <= total_rows < 10000:
                threshold = 0.02 * total_rows
            else:
                threshold = 0.01 * total_rows
            
            if unique_count <= threshold:
                categories = df[col].unique().tolist()
                column_details[col]['type'] = 'categorical'
                column_details[col]['categories'] = categories
            else:
                column_details[col]['type'] = 'other'
                column_details[col]['note'] = f"Too many unique values ({unique_count}) to list as categories."
                
    return column_details



In [5]:
# Preparing content from DB for prompting
query = "SELECT sql FROM sqlite_master WHERE type='table' AND name=?;"

# Using python
db_schema = []
table_details = []
for table_name in db_tables:
    print("-----------",table_name)
    # passing table name to pull their schema
    try:
        with sqlite3.connect(db_file) as conn:
            cursor = conn.cursor()
            # Table Schema
            cursor.execute(query, (table_name,))
            table_schema = cursor.fetchall()
            #print(table_schema[0])
            db_schema.append(table_schema[0])
            # Table details
            df = pd.read_sql_query(f"SELECT * FROM {table_name}", conn)
            details = get_column_details(df)

            for col_name, col_details in details.items():
                table_details.append((f"\n Table '{table_name}' where Column name is '{col_name}' contain details --> ",[f"{key}: {value}"for key, value in col_details.items()]))
                #print(f"\n Table '{table_name}' where Column name is '{col_name}' contain details --> ",[f"{key}: {value}"for key, value in col_details.items()])            
    
            # if table:
            #     # The result is a tuple, e.g., ('CREATE TABLE ...',)
            #     table[0]
            # else:
            #     print(f"Table '{table_name}' not found.")
                
    except sqlite3.Error as e:
        print(f"An error occurred: {e}")

final_db_schema = ';\n\n'.join(table[0] for table in db_schema)

----------- store
----------- store_return
----------- store_employee
----------- store_marketing


# Question start from here:---------------------------------------------------------

# Check is user query answerable as it is?

In [6]:
# Answer in Yes and No only to identify that user query can answer using single question or not
from pydantic import BaseModel

class GradeQuestion(BaseModel):
    """Binary score for hallucination present in generation answer."""
    binary_score: str = Field( description="Answer is grounded in the facts, 'yes' or 'no'" )
    
structured_llm_grader = model.with_structured_output(GradeQuestion)

Gradetemplate = """ You are a grader assessing whether user's question is suitable to generate sufficient prompt using below content: \n

You can use Database schema shared below to idenity possible outcomes. \n
You can also consider below table details while preparing sub-questions : {table_details} \n
You can join table 'store' with table 'store_return' using column 'Order ID', and there is one-to-one relationship between these two tables. \n
You can join table 'store' with table 'store_employee' using column 'Region', and there is many-to-one relationship between these two tables. \n
Column 'Monthly Incentive' in 'store_employee' is providing monhtly salary of employees. They can sell multiple products in a monthly but their monthly salary is fixed. \n
You can join table 'store' with table 'store_marketing' using column 'Campaign ID', and there is many-to-one relationship between these two tables. \n
Note: Give a binary score 'yes' or 'no'. 'Yes' means that question is suitable to generate sufficient prompt. \n
You can use database schema to identify available tables and data: {database_schema} \n

User Question: {question} \n
 """

prompt_grader = ChatPromptTemplate.from_template(Gradetemplate)

# Chain
question_grader = prompt_grader | structured_llm_grader

# Run
user_question = input()
grader = question_grader.invoke({"question":user_question, "table_details":table_details,"database_schema":final_db_schema})
print(grader.binary_score)

 what is the performance of the store finanacially?


yes


# Generate Multi Query

In [7]:
# Multi Query/Question if : user query can answer using single question, then we need Different Perspectives
# Decomposition
MultiQuerytemplate = """You are a helpful assistant and Act as a professional that generates multiple sub-questions related to an input question. \n
The goal is to break down the input into a set of sub-problems / sub-questions that can be answers in isolation. \n
You can use Database schema shared below to idenity possible outcomes. \n
You can join table 'store' with table 'store_return' using column 'Order ID', and there is one-to-one relationship between these two tables. \n
You can join table 'store' with table 'store_employee' using column 'Region', and there is many-to-one relationship between these two tables. \n
Column 'Monthly Incentive' in 'store_employee' is providing monhtly salary of employees. They can sell multiple products in a monthly but their monthly salary is fixed. \n
You can join table 'store' with table 'store_marketing' using column 'Campaign ID', and there is many-to-one relationship between these two tables. \n
You can also consider below notes while preparing sub-questions: \n
You can also consider table details while preparing the prompt : {table_details} \n
You can use database schema to identify available tables and data: {database_schema} \n
Generate multiple search queries related to: {question} \n
Output (generate minimum 1 and maximum 5 queries which are suffcient to answer the question) \n
Note: only provide queries in output, no other content \n
Important Note: Do not generate any SQL query, only prepare questions  """
prompt_decomposition = ChatPromptTemplate.from_template(MultiQuerytemplate)

# Chain
generate_queries_decomposition = ( prompt_decomposition | model | StrOutputParser() | (lambda x: x.split("\n")))

# Run
if grader.binary_score == 'yes':
    questions = [user_question]
    print("Single-Question: ",questions)
    
else:
    questions = generate_queries_decomposition.invoke({"question":user_question, "table_details":table_details,"database_schema":final_db_schema})  # for multiple question
    print("Multi-Questions: ",questions)
    

Single-Question:  ['what is the performance of the store finanacially?']


# # Prompt generation for SQL Code

In [8]:
# Prompt generation for SQL Code  - Question and Prompt Pairs
SQLprompt_template = """
You are an expert who understand user question and translate them in to right prompt using the database schema, where you can idneity right column name amd table name. \n
Database schema provided below, where each table's schema is separated by a semicolon (;). \n
Important Note:\n\n
Prompt should be detailed and include exact columns and tables name while writing the prompt. You can tak help of Database schema to identify the right column and table name. \n 
# You can consume database schema to identify the table columns name and table name. \n
# You can join table 'store' with table 'store_return' using column 'Order ID', and there is one-to-one relationship between these two tables. \n
# You can join table 'store' with table 'store_employee' using column 'Region', and there is many-to-one relationship between these two tables. \n
# Column 'Monthly Incentive' in 'store_employee' is providing monhtly salary of employees. They can sell multiple products in a monthly but their monthly salary is fixed. \n
# You can join table 'store' with table 'store_marketing' using column 'Campaign ID', and there is many-to-one relationship between these two tables. \n
# You can also consider table details while preparing the prompt : {table_details} \n

DO not join any table if there is no relation given in database schema or any other instruction provided. \n
Only provide prompt content which can further consume by other AI model. No other information required. \n
Note: Do not write SQL code, only provide prompt. \n

# database schema: {database_schema} \n
# User question: {question}
 """
response_prompt = ChatPromptTemplate.from_template(SQLprompt_template)

sql_prompt_chain1 =  response_prompt | model_flash1 | StrOutputParser() 
sql_prompt_chain2 =  response_prompt | model_flash2 | StrOutputParser() 


def que_prompt_pair(key,value):
    """Question and Prompt pair"""
    new_dict = {}
    new_dict[key] = value
    return new_dict

def Prompt_generation_for_SQL_Code(questions):
    final_ques_prompt_pair = []
    count = 0
    for que in questions:
        count += 1
        print(count,": ",que)
        if count < 10:
            sql_prompt_content = sql_prompt_chain1.invoke({"question": que, "database_schema":final_db_schema, "table_details":table_details})
        elif count >= 10 and count < 19:
            sql_prompt_content = sql_prompt_chain2.invoke({"question": que, "database_schema":final_db_schema, "table_details":table_details})
        else:
            break
            
        que_prompt_dict = que_prompt_pair(que, sql_prompt_content)
        final_ques_prompt_pair.append(que_prompt_dict)
        #print(final_ques_prompt_pair)
    
    return final_ques_prompt_pair

# As an expert SQL coder, your task is to generate a single, accurate SQL query (or a multi-step query executing in one shot) based on the user's {question}. 
# Only provide the SQL code in your response, with no additional text or explanation.
# Close the query in triple quotes (''') only and do not include any special charactor.

In [9]:
final_ques_prompt_pair = Prompt_generation_for_SQL_Code(questions)
print(final_ques_prompt_pair)

1 :  what is the performance of the store finanacially?
[{'what is the performance of the store finanacially?': "Please provide the total 'Sales' and total 'Profit' from the 'store' table to assess the financial performance of the store."}]


# Check hallucination

In [10]:
# Check hallucination - Answer in Yes and No | Getting Error fetching the URL: 'NoneType' object has no attribute 'binary_score'

generated_prompt = []
for i in final_ques_prompt_pair:
    generated_prompt.append(list(i.values())[0])
    sql_prompt_content = list(i.values())[0]
    # print(list(i.values())[0])
    
class GradeHallucinations(BaseModel):
    """Binary score for hallucination present in generated prompt."""

    binary_score: str = Field(
        description="Answer is grounded in the facts, 'yes' or 'no'"
    )

# LLM with function call
structured_llm_grader = model.with_structured_output(GradeHallucinations)

# Prompt
system = """You are a grader assessing whether this 'set of prompts' are capable to answer the question. \n 
     Give a binary score 'yes' or 'no'. 'Yes' means that the set of prompts are capable to answer the question. 'No' means that the set of prompts are not capable to answer the question."""

hallucination_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "Question: {Question} \n\n Set of Prompt: {Generated_Prompt}"),
    ]
)

hallucination_grader = hallucination_prompt | structured_llm_grader

try:
    hallucination_check = hallucination_grader.invoke({"Question": user_question, "Generated_Prompt": generated_prompt})
    if hallucination_check.binary_score == 'Yes':
        print("There is no hallucination.")
    else:
        print("There is a hallucination, need to re think the answer.")
except Exception as e:
    print(f"Error fetching the URL: {e}")
    

There is a hallucination, need to re think the answer.


In [ ]:
# Re-Think

In [12]:
# If there is hallucination, then re-think the answer
import json
def extract_questions_from_json(data):
    questions = []
    
    if isinstance(data, dict):
        # Iterate over values in the dictionary
        for value in data.values():
            # If the value is a list or another dictionary, recurse
            if isinstance(value, (dict, list)):
                questions.extend(extract_questions_from_json(value))
            # If the value is a string, it's a question
            elif isinstance(value, str):
                questions.append(value)
    
    elif isinstance(data, list):
        # Iterate over items in the list
        for item in data:
            # If the item is a string, it's a question
            if isinstance(item, str):
                questions.append(item)
            # If the item is a dict, recurse
            elif isinstance(item, dict):
                questions.extend(extract_questions_from_json(item))
                
    return questions
    
from langchain_core.output_parsers import JsonOutputParser
parser = JsonOutputParser()

system = """ You are a experienced professional who understand business and finance and capable to think the possible objective of user's question based on available data content and generate outlines which could help to find the solution. Previously genetared prompt is not capable to answer \n.
Important Note: Do not generate any SQL query, only prepare outlines in category and sub-category format if required which can help to figureout the answer. \n
Output must be restricted in json format. and follow below schema:

"""


data_contnet = """
You can use Database schema shared below to idenity possible outcomes. \n
You can join table 'store' with table 'store_return' using column 'Order ID', and there is one-to-one relationship between these two tables. \n
You can join table 'store' with table 'store_employee' using column 'Region', and there is many-to-one relationship between these two tables. \n
Column 'Monthly Incentive' in 'store_employee' is providing monhtly salary of employees. They can sell multiple products in a monthly but their monthly salary is fixed. \n
You can join table 'store' with table 'store_marketing' using column 'Campaign ID', and there is many-to-one relationship between these two tables. \n
"""

rethink_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "Question: {Question} \n\n Previously generated Set of Prompt: {Previously_Generated_Prompt}"),
        ("human", "database schema: {database_schema} \n\n  table data details  : {table_details}  \n\n Other Content : {data_contnet}"),
    ]
)


# def rethink_outlines_func():
rethink_chain = rethink_prompt | model | parser
try:
    rethink_outcomes = rethink_chain.invoke({"Question": user_question, "Previously_Generated_Prompt": generated_prompt, "data_contnet": data_contnet, "table_details":table_details, "database_schema":final_db_schema })
    print(rethink_outcomes)
except Exception as e:
    print(f"Error fetching the URL: {e}")

rethink_questions = json.dumps(extract_questions_from_json(rethink_outcomes))
print("*"*90)
print(json.dumps(rethink_questions))

    # return rethink_questions



{'outline': {'Financial Performance': {'Overall Metrics': {'Total Sales': "Calculate the sum of the 'Sales' column in the 'store' table.", 'Total Profit': "Calculate the sum of the 'Profit' column in the 'store' table.", 'Profit Margin': 'Calculate (Total Profit / Total Sales) * 100.'}, 'Performance by Category': {'Sales by Category': "Group the 'store' table by 'Category' and sum the 'Sales' for each category.", 'Profit by Category': "Group the 'store' table by 'Category' and sum the 'Profit' for each category.", 'Profit Margin by Category': 'For each category, calculate (Sum of Profit / Sum of Sales) * 100.'}, 'Performance by Sub-Category': {'Sales by Sub-Category': "Group the 'store' table by 'Sub-Category' and sum the 'Sales' for each sub-category.", 'Profit by Sub-Category': "Group the 'store' table by 'Sub-Category' and sum the 'Profit' for each sub-category.", 'Profit Margin by Sub-Category': 'For each sub-category, calculate (Sum of Profit / Sum of Sales) * 100.'}, 'Performance

In [13]:
json_questions_template = """
You are and export and you have to pull all questions from given json data and output should content only a list of questions not other information. \n
Seperate each question by new line. \n
Json data : {json_data}

 """
json_questions_prompt = ChatPromptTemplate.from_template(json_questions_template)

json_questions_chain =  json_questions_prompt | model | StrOutputParser() 
json_questions_output = json_questions_chain.invoke({"json_data":rethink_questions})
json_questions = [i for i in json_questions_output.split("\n")]
print(json_questions)

["Calculate the sum of the 'Sales' column in the 'store' table.", "Calculate the sum of the 'Profit' column in the 'store' table.", 'Calculate (Total Profit / Total Sales) * 100.', "Group the 'store' table by 'Category' and sum the 'Sales' for each category.", "Group the 'store' table by 'Category' and sum the 'Profit' for each category.", 'For each category, calculate (Sum of Profit / Sum of Sales) * 100.', "Group the 'store' table by 'Sub-Category' and sum the 'Sales' for each sub-category.", "Group the 'store' table by 'Sub-Category' and sum the 'Profit' for each sub-category.", 'For each sub-category, calculate (Sum of Profit / Sum of Sales) * 100.', "Group the 'store' table by 'Region' and sum the 'Sales' for each region.", "Group the 'store' table by 'Region' and sum the 'Profit' for each region.", 'For each region, calculate (Sum of Profit / Sum of Sales) * 100.', "Group the 'store' table by 'Segment' and sum the 'Sales' for each segment.", "Group the 'store' table by 'Segment' 

In [ ]:
# if hallucination_check.binary_score == 'Yes':
#     questions = questions
#     print("------------------ No Hallucination -------------------")
#     final_ques_prompt_pair = final_ques_prompt_pair
# else:
#     print("------------------ Re Thinking -------------------")
#     questions = rethink_outlines_func()
#     print("------------------ Generating new Questions  -------------------")
#     final_ques_prompt_pair = Prompt_generation_for_SQL_Code(json_questions)

In [14]:
final_ques_prompt_pair = Prompt_generation_for_SQL_Code(json_questions)


1 :  Calculate the sum of the 'Sales' column in the 'store' table.
2 :  Calculate the sum of the 'Profit' column in the 'store' table.
3 :  Calculate (Total Profit / Total Sales) * 100.
4 :  Group the 'store' table by 'Category' and sum the 'Sales' for each category.
5 :  Group the 'store' table by 'Category' and sum the 'Profit' for each category.
6 :  For each category, calculate (Sum of Profit / Sum of Sales) * 100.
7 :  Group the 'store' table by 'Sub-Category' and sum the 'Sales' for each sub-category.
8 :  Group the 'store' table by 'Sub-Category' and sum the 'Profit' for each sub-category.
9 :  For each sub-category, calculate (Sum of Profit / Sum of Sales) * 100.
10 :  Group the 'store' table by 'Region' and sum the 'Sales' for each region.
11 :  Group the 'store' table by 'Region' and sum the 'Profit' for each region.
12 :  For each region, calculate (Sum of Profit / Sum of Sales) * 100.
13 :  Group the 'store' table by 'Segment' and sum the 'Sales' for each segment.
14 :  Gro

# generate SQL Code

In [15]:
# This will generate SQL Code - Question and SQL code pairs

sql_query_prompt_template = """
As an expert SQL coder, your task is to generate a single, accurate SQL query (or a multi-step query executing in one shot) based on the user's question. \n
You can consume below share context to write correct SQL query. \n
You can consume database schema to identify the table columns name and table name. \n
DO not join any table if there is no relation or any other details provided. \n
You can join table 'store' with table 'store_return' using column 'Order ID', and there is one-to-one relationship between these two tables. \n
You can join table 'store' with table 'store_employee' using column 'Region', and there is many-to-one relationship between these two tables. \n
Column 'Monthly Incentive' in 'store_employee' is providing monhtly salary of employees. They can sell multiple products in a monthly but their monthly salary is fixed. \n
You can join table 'store' with table 'store_marketing' using column 'Campaign ID', and there is many-to-one relationship between these two tables. \n
If you are pulling multiple values in output, so that output should be sorted by numerical values. If numeircal value is not there then by categorical column. \n
You can also consider table details while preparing the prompt : {table_details} \n

Note: Only provide the SQL code in your response. No additional text or explanation. \n

# SQL Query Content : {query_content} \n\n
# database schema: {database_schema} \n\n
 """
query_prompt = ChatPromptTemplate.from_template(sql_query_prompt_template)

sql_code_chain1 =  query_prompt | model_flash1 | StrOutputParser()
sql_code_chain2 =  query_prompt | model_flash2 | StrOutputParser()

# sql_code_content = sql_prompt_chain.invoke({"database_schema":final_db_schema,"query_content":sql_prompt_content})


final_ques_code_pair = []
count = 0
for i in final_ques_prompt_pair:
    que = list(i.keys())[0]
    sql_prompt_content = list(i.values())[0]
    #print(list(i.keys())[0])
    #print(list(i.values())[0])
    count += 1
    print(count,": ",que)
    if count < 10:
        sql_code_content = sql_code_chain1.invoke({"database_schema":final_db_schema,"query_content":sql_prompt_content, "table_details":table_details, "database_schema":final_db_schema})
    elif count >= 10 and count < 19:
        sql_code_content = sql_code_chain2.invoke({"database_schema":final_db_schema,"query_content":sql_prompt_content, "table_details":table_details, "database_schema":final_db_schema})
    else:
        break
    que_code_dict = que_prompt_pair(que, sql_code_content)
    final_ques_code_pair.append(que_code_dict)    





1 :  Calculate the sum of the 'Sales' column in the 'store' table.
2 :  Calculate the sum of the 'Profit' column in the 'store' table.
3 :  Calculate (Total Profit / Total Sales) * 100.
4 :  Group the 'store' table by 'Category' and sum the 'Sales' for each category.
5 :  Group the 'store' table by 'Category' and sum the 'Profit' for each category.
6 :  For each category, calculate (Sum of Profit / Sum of Sales) * 100.
7 :  Group the 'store' table by 'Sub-Category' and sum the 'Sales' for each sub-category.
8 :  Group the 'store' table by 'Sub-Category' and sum the 'Profit' for each sub-category.
9 :  For each sub-category, calculate (Sum of Profit / Sum of Sales) * 100.
10 :  Group the 'store' table by 'Region' and sum the 'Sales' for each region.
11 :  Group the 'store' table by 'Region' and sum the 'Profit' for each region.
12 :  For each region, calculate (Sum of Profit / Sum of Sales) * 100.
13 :  Group the 'store' table by 'Segment' and sum the 'Sales' for each segment.
14 :  Gro

# Execute SQL Code

In [16]:
# This section execute SQL code to pull requested data and show as output - no data ingestion in AI model
from tabulate import tabulate

question_answer_pair = []

for i in final_ques_code_pair:
    #print(i)
    que = list(i.keys())[0]
    sql_string = list(i.values())[0]
    print("Q.",list(i.keys())[0])
    #print(list(i.values())[0])


    #sql_string =  output #'''SELECT Returned, COUNT('Order ID') FROM store_return GROUP BY Returned''' # output
    
    # 1. Remove optional "sql" keyword at the beginning
    cleaned_sql = sql_string.lstrip("sql").strip()
    
    # 2. Remove triple backticks from markdown code blocks
    cleaned_sql = cleaned_sql.strip('`')
    
    # Remove the optional markdown language identifier
    if cleaned_sql.startswith('sql'):
        cleaned_sql = cleaned_sql[3:].strip()
    
    db_file = "Chinook.db"
    # Using SQL
    try:
        with sqlite3.connect(db_file) as conn:
            cursor = conn.cursor()
            cursor.execute(cleaned_sql)                  #executescript(sql_query)           #execute(sql_query)  for single query
            records = cursor.fetchall()  # fetch all data from table
            #print(records)
            if pd.DataFrame(records).shape[0]>1 or pd.DataFrame(records).shape[1]>1:
                answer_print = tabulate(pd.DataFrame(records), headers= [description[0] for description in cursor.description], tablefmt='fancy_grid')
                answer = pd.DataFrame(records,columns=[description[0] for description in cursor.description])
            else:
                try:
                    answer_print = f"{records[0][0] : .2f}"
                    answer = f"{records[0][0] : .2f}"
                except Exception as e:
                    print(f"Error fetching the URL: {e}")

            print(answer_print)
            question_answer_dict = que_prompt_pair(que, answer)
            question_answer_pair.append(question_answer_dict)  

    except sqlite3.Error as e:
        print(f"An error occurred: {e}")
    print("\n","*"*105)

Q. Calculate the sum of the 'Sales' column in the 'store' table.
 2297200.86

 *********************************************************************************************************
Q. Calculate the sum of the 'Profit' column in the 'store' table.
 286397.02

 *********************************************************************************************************
Q. Calculate (Total Profit / Total Sales) * 100.
 12.47

 *********************************************************************************************************
Q. Group the 'store' table by 'Category' and sum the 'Sales' for each category.
╒════╤═════════════════╤══════════════╕
│    │ Category        │   TotalSales │
╞════╪═════════════════╪══════════════╡
│  0 │ Technology      │       836154 │
├────┼─────────────────┼──────────────┤
│  1 │ Furniture       │       742000 │
├────┼─────────────────┼──────────────┤
│  2 │ Office Supplies │       719047 │
╘════╧═════════════════╧══════════════╛

 ************************

# Pass Outline with Question & Answer to prepare a analytical report

In [19]:
analyst_prompt_template = """
You are an export and professional Data Analyst and you task is to prepare a analytica report using the outlines and relevent questions answers provided by user. \n
Report should be crisp and professional, and should show a anaytical story. \n
# Outlines in json format : {rethink_outcomes} \n
# Questions & Answer Pairs: {question_answer_pair} \n
 """
analyst_prompt = ChatPromptTemplate.from_template(analyst_prompt_template)

analyst_code_chain =  analyst_prompt | model_flash1 | StrOutputParser()

analyst_result = analyst_code_chain.invoke({"rethink_outcomes":rethink_outcomes,"question_answer_pair":question_answer_pair})
print(analyst_result)


## Data Analysis Report: Financial Performance Overview

**Date:** October 26, 2023
**Prepared For:** Management Team
**Prepared By:** Data Analytics Department

---

### 1. Executive Summary

This report provides a comprehensive analysis of the company's financial performance, dissecting key metrics across categories, sub-categories, regions, and customer segments. The overall profit margin stands at **12.47%** on total sales of **$2.30 million**.

Key findings highlight that **Technology** is the leading category in both sales and profit, maintaining a strong profit margin. Conversely, **Furniture** shows a significantly lower profit margin of **2.49%**, primarily dragged down by sub-categories like **Tables** and **Bookcases** which are operating at a loss. Geographically, the **West** and **East** regions are top performers in both sales and profitability, while the **Central** region lags with the lowest profit margin.

Customer segmentation reveals that **Home Office** customers,